### Workflow

Compare BPOSD with forced OSD against Tesseract on the selected one-round/perfect-measurement settings. The purpose is to test whether disagreements remain after forcing OSD, not to infer that every remaining disagreement has the same cause.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/force_osd_failures'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/paths.json')


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

DATA_DIR = _data_paths.run()
DATA_DIR.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations
from utils.misc import *
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract
from utils.graph_vis import visualize_graph_plotly, flatten_graph
from utils.graph_struct import build_tanner_graph_from_dem, graph_struct
from beliefmatching import detector_error_model_to_check_matrices
from utils.sample import count_samples_union
from matplotlib_venn import venn2
from utils.sample import collect_samples_with_errors
from src.decoders import CythonBPOSD
from utils.belief_tracking import plot_belief_tracking, plot_history_heatmap
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import animate_hard_decisions_on_graph, visualize_det_on_tanner_graph, visualize_graph_plotly


In [ ]:
import numpy as np

noise_model = 'circuit_level'
d_rounds = 1
max_shots = 1000000000
max_errors = 2000
p_values = np.geomspace(1e-08, 0.1, 12)


In [ ]:
from experiment.utils import benchmark_plotting

stats_to_df = benchmark_plotting.stats_to_df
plot_gap_interval = benchmark_plotting.plot_gap_interval
save_file = _data_paths.input('2026_05_28_circuit_level_1_perfect_meas_shots_1000000000_csv')
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > 0].sort_values(['code', 'p'])
points


# Force OSD

## Sample with errors

### Collect the forced-OSD outcome groups

The next loops retain target disagreements, both-success controls, reverse disagreements and both-failure cases using the forced-OSD decoder pair. Keep their acquisition settings and selection limits with the saved rows.


In [ ]:
from utils.sample import collect_samples_with_errors

CASES_DIR = _data_paths.output('force_OSD')
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
cases_path = _data_paths.output('bfts_cases_csv')
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        build_decoders='force_osd',
        kind='bposd_fail_tess_success',
    )

    print(f'collected {len(recs)} failure cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bs_cases_csv')
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        build_decoders='force_osd',
        kind='both_success',
    )

    print(f'collected {len(recs)} both_success cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bstf_cases_csv')
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        build_decoders='force_osd',
        kind='bposd_success_tess_fail',
    )

    print(f'collected {len(recs)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bf_cases_csv')
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        build_decoders='force_osd',
        kind='both_fail',
    )

    print(f'collected {len(recs)} both_fail cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


### Reload the acquired cases

Read the saved bit strings and physical-error fields for the distribution comparisons below. These are selected example tables, so their relative sizes are not an unbiased LER comparison.


In [ ]:
types = {
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

bf_cases = pd.read_csv(_data_paths.output('bf_cases_csv'), dtype=types)
bs_cases = pd.read_csv(_data_paths.output('bs_cases_csv'), dtype=types)
bfts_cases = pd.read_csv(_data_paths.output('bfts_cases_csv'), dtype=types)
bstf_cases = pd.read_csv(_data_paths.output('bstf_cases_csv'), dtype=types)


### Error Weight Distribution

CA_5_2_3

### Compare error weights by code

For each fixed code, compare error-weight counts and fractions across outcome groups. Observing disagreements in both directions demonstrates that forced OSD does not make the two decoders equivalent on these samples.


In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_5_2_3', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)


CA_9_2_5

In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_9_2_5', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)


### Inspect dependence on p

Resolve the target disagreement cases by p and error weight before reading the plotted distribution. Differences in retained samples should not be interpreted as a controlled mechanism intervention.


In [ ]:
dist = bfts_cases[bfts_cases['code'] == 'CA_9_2_5'].groupby(['p', 'error_weight']).size().reset_index(name='count')
dist['prob'] = dist.groupby('p')['count'].transform(lambda x: x / x.sum())
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_9_2_5 in different p (bfts)', fontsize=12)

for i, (p_value, sub) in enumerate(dist.groupby('p')):
    axes[0].bar(sub['error_weight'] + i * 0.1, sub['count'], label=f'p={p_value:.6f}', width=0.1)
    axes[1].bar(sub['error_weight'] + i * 0.1, sub['prob'], label=f'p={p_value:.6f}', width=0.1)

min_w = dist['error_weight'].min()
max_w = dist['error_weight'].max()
axes[0].set_xticks(range(min_w, max_w + 1, 1))
axes[0].set_xlabel('error_weight')
axes[0].set_ylabel('count')
axes[0].set_title('count vs error weight')
axes[0].legend()
axes[1].set_xticks(range(min_w, max_w + 1, 1))
axes[1].set_xlabel('error_weight')
axes[1].set_ylabel('percentage')
axes[1].set_title('percentage vs error weight')
axes[1].legend()
plt.tight_layout()

plt.show()


In [ ]:
bfts_cases[bfts_cases['code'] == 'CA_9_2_5']


In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.violinplot(
    data=bfts_cases[bfts_cases['code'] == 'CA_9_2_5'],
    x='p',
    y='error_llr_weight',
    inner='box',
)
labels = [f'{float(t.get_text()):.6e}' for t in ax.get_xticklabels()]
ax.set_xticklabels(labels, rotation=45)
plt.xticks(rotation=45)
plt.xlabel('p')
plt.ylabel('error_llr_weight')
plt.title('Distribution of error_llr_weight for different p')
plt.tight_layout()

plt.show()
